In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Big Data Analysis") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.1.2


In [2]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [3]:
df = spark.read.csv(
    "OnlineRetail.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
only showing top 5 rows


In [4]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



In [5]:
print("Rows :", df.count())
print("Columns :", len(df.columns))

Rows : 541909
Columns : 8


In [6]:
from pyspark.sql.functions import *

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|       1454|       0|          0|        0|    135080|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+



In [7]:
from pyspark.sql.functions import col

df = df.withColumn(
    "TotalAmount",
    col("Quantity") * col("UnitPrice")
)

df.show(5)

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|       TotalAmount|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+------------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|15.299999999999999|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|             20.34|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|              22.0|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|             20.34|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|             20.34|
+---------+---------+---

In [8]:
df.groupBy("Description") \
  .agg(sum("Quantity").alias("Total Sold")) \
  .orderBy(desc("Total Sold")) \
  .show(10, False)

+----------------------------------+----------+
|Description                       |Total Sold|
+----------------------------------+----------+
|WORLD WAR 2 GLIDERS ASSTD DESIGNS |53847     |
|JUMBO BAG RED RETROSPOT           |47363     |
|ASSORTED COLOUR BIRD ORNAMENT     |36381     |
|POPCORN HOLDER                    |36334     |
|PACK OF 72 RETROSPOT CAKE CASES   |36039     |
|WHITE HANGING HEART T-LIGHT HOLDER|35317     |
|RABBIT NIGHT LIGHT                |30680     |
|MINI PAINT SET VINTAGE            |26437     |
|PACK OF 12 LONDON TISSUES         |26315     |
|PACK OF 60 PINK PAISLEY CAKE CASES|24753     |
+----------------------------------+----------+
only showing top 10 rows


In [9]:
df.groupBy("Country") \
  .agg(sum("TotalAmount").alias("Revenue")) \
  .orderBy(desc("Revenue")) \
  .show(10, False)

+--------------+------------------+
|Country       |Revenue           |
+--------------+------------------+
|United Kingdom|8187806.363998182 |
|Netherlands   |284661.5399999997 |
|EIRE          |263276.8199999998 |
|Germany       |221698.21000000066|
|France        |197403.90000000008|
|Australia     |137077.2699999999 |
|Switzerland   |56385.3500000001  |
|Spain         |54774.58000000004 |
|Belgium       |40910.95999999999 |
|Sweden        |36595.90999999999 |
+--------------+------------------+
only showing top 10 rows


In [10]:
df.groupBy("CustomerID") \
  .agg(sum("TotalAmount").alias("Total Spending")) \
  .orderBy(desc("Total Spending")) \
  .show(10)

+----------+------------------+
|CustomerID|    Total Spending|
+----------+------------------+
|      NULL|1447682.1200001389|
|     14646| 279489.0199999998|
|     18102|256438.48999999993|
|     17450|187482.16999999993|
|     14911|132572.62000000017|
|     12415|123725.44999999998|
|     14156| 113384.1400000001|
|     17511| 88125.37999999999|
|     16684| 65892.07999999999|
|     13694|62653.099999999984|
+----------+------------------+
only showing top 10 rows


In [11]:
from pyspark.sql.functions import count

df.groupBy("Description") \
  .agg(count("*").alias("Orders")) \
  .orderBy(desc("Orders")) \
  .show(10, False)

+----------------------------------+------+
|Description                       |Orders|
+----------------------------------+------+
|WHITE HANGING HEART T-LIGHT HOLDER|2369  |
|REGENCY CAKESTAND 3 TIER          |2200  |
|JUMBO BAG RED RETROSPOT           |2159  |
|PARTY BUNTING                     |1727  |
|LUNCH BAG RED RETROSPOT           |1638  |
|ASSORTED COLOUR BIRD ORNAMENT     |1501  |
|SET OF 3 CAKE TINS PANTRY DESIGN  |1473  |
|NULL                              |1454  |
|PACK OF 72 RETROSPOT CAKE CASES   |1385  |
|LUNCH BAG  BLACK SKULL.           |1350  |
+----------------------------------+------+
only showing top 10 rows


In [14]:
from pyspark.sql.functions import countDistinct, desc

In [15]:
df.groupBy("Country") \
  .agg(countDistinct("InvoiceNo").alias("Invoices")) \
  .orderBy(desc("Invoices")) \
  .show(10)

+--------------+--------+
|       Country|Invoices|
+--------------+--------+
|United Kingdom|   23494|
|       Germany|     603|
|        France|     461|
|          EIRE|     360|
|       Belgium|     119|
|         Spain|     105|
|   Netherlands|     101|
|   Switzerland|      74|
|      Portugal|      71|
|     Australia|      69|
+--------------+--------+
only showing top 10 rows


In [16]:
df.createOrReplaceTempView("retail")

In [17]:
spark.sql("""
SELECT
    Country,
    ROUND(SUM(TotalAmount), 2) AS Revenue
FROM retail
GROUP BY Country
ORDER BY Revenue DESC
LIMIT 10
""").show()

+--------------+----------+
|       Country|   Revenue|
+--------------+----------+
|United Kingdom|8187806.36|
|   Netherlands| 284661.54|
|          EIRE| 263276.82|
|       Germany| 221698.21|
|        France|  197403.9|
|     Australia| 137077.27|
|   Switzerland|  56385.35|
|         Spain|  54774.58|
|       Belgium|  40910.96|
|        Sweden|  36595.91|
+--------------+----------+



In [18]:
spark.sql("""
SELECT
    Description,
    SUM(Quantity) AS TotalSold
FROM retail
GROUP BY Description
ORDER BY TotalSold DESC
LIMIT 10
""").show(truncate=False)

+----------------------------------+---------+
|Description                       |TotalSold|
+----------------------------------+---------+
|WORLD WAR 2 GLIDERS ASSTD DESIGNS |53847    |
|JUMBO BAG RED RETROSPOT           |47363    |
|ASSORTED COLOUR BIRD ORNAMENT     |36381    |
|POPCORN HOLDER                    |36334    |
|PACK OF 72 RETROSPOT CAKE CASES   |36039    |
|WHITE HANGING HEART T-LIGHT HOLDER|35317    |
|RABBIT NIGHT LIGHT                |30680    |
|MINI PAINT SET VINTAGE            |26437    |
|PACK OF 12 LONDON TISSUES         |26315    |
|PACK OF 60 PINK PAISLEY CAKE CASES|24753    |
+----------------------------------+---------+



In [19]:
spark.sql("""
SELECT
    CustomerID,
    ROUND(SUM(TotalAmount), 2) AS Spending
FROM retail
WHERE CustomerID IS NOT NULL
GROUP BY CustomerID
ORDER BY Spending DESC
LIMIT 10
""").show()

+----------+---------+
|CustomerID| Spending|
+----------+---------+
|     14646|279489.02|
|     18102|256438.49|
|     17450|187482.17|
|     14911|132572.62|
|     12415|123725.45|
|     14156|113384.14|
|     17511| 88125.38|
|     16684| 65892.08|
|     13694|  62653.1|
|     15311| 59419.34|
+----------+---------+



In [20]:
spark.sql("""
SELECT
    Description,
    ROUND(AVG(UnitPrice), 2) AS AvgPrice
FROM retail
GROUP BY Description
ORDER BY AvgPrice DESC
LIMIT 10
""").show(truncate=False)

+----------------------------------+--------+
|Description                       |AvgPrice|
+----------------------------------+--------+
|AMAZON FEE                        |7324.78 |
|PICNIC BASKET WICKER 60 PIECES    |649.5   |
|CRUK Commission                   |495.84  |
|Manual                            |374.91  |
|DOTCOM POSTAGE                    |290.91  |
|Bank Charges                      |202.86  |
|REGENCY MIRROR WITH SHUTTERS      |156.43  |
|RUSTIC  SEVENTEEN DRAWER SIDEBOARD|156.03  |
|VINTAGE RED KITCHEN CABINET       |150.66  |
|VINTAGE BLUE KITCHEN CABINET      |143.65  |
+----------------------------------+--------+



In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Big Data Analysis") \
    .getOrCreate()

In [8]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    when,
    sum,
    avg,
    max,
    min,
    desc
)

In [9]:
df = spark.read.csv(
    "OnlineRetail.csv",
    header=True,
    inferSchema=True
)

df = df.withColumn(
    "TotalAmount",
    col("Quantity") * col("UnitPrice")
)

In [10]:
import pandas as pd

pdf = df.toPandas()

pdf.to_csv("Processed_OnlineRetail.csv", index=False)

print("✅ CSV Saved Successfully!")

✅ CSV Saved Successfully!


In [11]:
spark.stop()

print("🎉 Big Data Analysis Project Completed Successfully!")

🎉 Big Data Analysis Project Completed Successfully!


In [12]:
import os

print(os.listdir())

['.ipynb_checkpoints', '.jupyter', 'BigDataAnalysis.ipynb', 'OnlineRetail.csv', 'Processed_OnlineRetail', 'Processed_OnlineRetail.csv', 'Untitled.ipynb']
